In [6]:
import math
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset, ConcatDataset

from torchvision import datasets, transforms

In [7]:
img_size = 28

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

batch_size = 128
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, groups=8):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),  # GroupNorm is standard for Diffusion
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class uNet(nn.Module):
    def __init__(self, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input = DoubleConv(1, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))

        self.bottleneck = DoubleConv(256, 512)

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(384, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(192, 128)

        self.output = nn.Conv2d(128, 1, kernel_size=1)

        self.emb_down1 = nn.Linear(time_dim, 128)
        self.emb_down2 = nn.Linear(time_dim, 256)
        self.emb_up1 = nn.Linear(time_dim, 256)
        self.emb_up2 = nn.Linear(time_dim, 128)

    def forward(self, x, t):
        t = t.view(-1, 1) 
        emb = self.time_mlp(t)
        
        x1 = self.input(x)
        
        x2 = self.down1(x1)
        x2 = x2 + self.emb_down1(emb)[:, :, None, None]
        
        x3 = self.down2(x2)
        x3 = x3 + self.emb_down2(emb)[:, :, None, None]

        x_mid = self.bottleneck(x3)

        x_up1 = self.up1(x_mid)
        x_up1 = torch.cat([x_up1, x2], dim=1)
        x_up1 = self.conv_up1(x_up1) 
        x_up1 = x_up1 + self.emb_up1(emb)[:, :, None, None]
        
        x_up2 = self.up2(x_up1)
        x_up2 = torch.cat([x_up2, x1], dim=1)
        x_up2 = self.conv_up2(x_up2)
        x_up2 = x_up2 + self.emb_up2(emb)[:, :, None, None]

        return self.output(x_up2)
model = uNet().to("cpu")
print(sum(p.numel() for p in model.parameters()))
del model

7302593


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 28
batch_size = 128
epochs = 100
lr = 1e-3
timesteps = 500
model = uNet().to(device)
p_uncond = 0.2

In [5]:
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

beta = torch.linspace(1e-4, 0.02, timesteps).to(device)
alpha = 1. - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def forward_diffusion(x0, t):
    noise = torch.randn_like(x0)
    sqrt_alpha_bar = torch.sqrt(alpha_bar[t])[:, None, None, None]
    sqrt_one_minus_alpha_bar = torch.sqrt(1 - alpha_bar[t])[:, None, None, None]
    return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise, noise

print("Starting Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        
        t = torch.randint(0, timesteps, (images.size(0),), device=device)
        
        noisy_imgs, noise = forward_diffusion(images, t)
        
        noise_pred = model(noisy_imgs, t.float() / timesteps)
        
        loss = criterion(noise_pred, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    print("Generating Samples...")
    model.eval()
    with torch.no_grad():
        x = torch.randn(16, 1, img_size, img_size).to(device)
        
        for i in reversed(range(timesteps)):
            t = torch.tensor([i] * 16, device=device)
            
            predicted_noise = model(x, t.float() / timesteps)
            
            curr_alpha = alpha[i]
            curr_alpha_bar = alpha_bar[i]
            curr_beta = beta[i]
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            x = (1 / torch.sqrt(curr_alpha)) * (x - ((1 - curr_alpha) / (torch.sqrt(1 - curr_alpha_bar))) * predicted_noise) + torch.sqrt(curr_beta) * noise
    
        x = x.clamp(-1, 1) * 0.5 + 0.5
        x = x.cpu()
        
        grid_img = torch.cat([x[i] for i in range(8)], dim=2)
        plt.figure(figsize=(10, 2))
        plt.imshow(grid_img.squeeze(), cmap='gray')
        plt.axis('off')
        plt.title("Generated Images")
        plt.show()

Starting Training...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000002584F068C10>
Traceback (most recent call last):
  File "C:\Users\Sam\anaconda3\envs\UNI_GROUP_PROJECT\lib\site-packages\torch\utils\data\dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "C:\Users\Sam\anaconda3\envs\UNI_GROUP_PROJECT\lib\site-packages\torch\utils\data\dataloader.py", line 1628, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "C:\Users\Sam\anaconda3\envs\UNI_GROUP_PROJECT\lib\multiprocessing\process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "C:\Users\Sam\anaconda3\envs\UNI_GROUP_PROJECT\lib\multiprocessing\popen_spawn_win32.py", line 108, in wait
    res = _winapi.WaitForSingleObject(int(self._handle), msecs)
KeyboardInterrupt: 

KeyboardInterrupt



In [10]:
# torch.save(model.state_dict(), "uncond_diff_wei.pth")
j = 0

In [11]:
model.load_state_dict(torch.load("uncond_diff_wei.pth", map_location=device))
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

beta = torch.linspace(1e-4, 0.02, timesteps).to(device)
alpha = 1. - beta
alpha_bar = torch.cumprod(alpha, dim=0)

n_samples = 1000

for j in range(5):
    
    print("Generating Samples...")
    model.eval()
    with torch.no_grad():
        x = torch.randn(n_samples, 1, img_size, img_size).to(device)
        
        for i in reversed(range(timesteps)):
            if i % 100 == 0:
                print(i)
            t = torch.tensor([i] * n_samples, device=device)
            
            predicted_noise = model(x, t.float() / timesteps)
            
            curr_alpha = alpha[i]
            curr_alpha_bar = alpha_bar[i]
            curr_beta = beta[i]
            
            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            x = (1 / torch.sqrt(curr_alpha)) * (x - ((1 - curr_alpha) / (torch.sqrt(1 - curr_alpha_bar))) * predicted_noise) + torch.sqrt(curr_beta) * noise
    
        x = x.clamp(-1, 1) * 0.5 + 0.5
        x = x.cpu()
        
        import os
        from torchvision.utils import save_image
        
        os.makedirs(r"fake_mnist/train/fake", exist_ok=True)
        
        last = n_samples * j
        for i in range(x.size(0)):
            save_image(x[i], f"fake_mnist/train/fake/{i+last:05d}.png")
        print(i)

Generating Samples...
400
300
200
100
0
999
Generating Samples...
400
300
200
100
0
999
Generating Samples...
400
300
200
100
0
999
Generating Samples...
400
300
200
100
0
999
Generating Samples...
400
300
200
100
0
999


In [13]:
model.load_state_dict(torch.load("uncond_diff_wei.pth", map_location=device))
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

beta = torch.linspace(1e-4, 0.02, timesteps).to(device)
alpha = 1. - beta
alpha_bar = torch.cumprod(alpha, dim=0)

n_samples = 1000

print("Generating Samples...")
model.eval()
with torch.no_grad():
    x = torch.randn(n_samples, 1, img_size, img_size).to(device)
    
    for i in reversed(range(timesteps)):
        if i % 100 == 0:
            print(i)
        t = torch.tensor([i] * n_samples, device=device)
        
        predicted_noise = model(x, t.float() / timesteps)
        
        curr_alpha = alpha[i]
        curr_alpha_bar = alpha_bar[i]
        curr_beta = beta[i]
        
        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)
            
        x = (1 / torch.sqrt(curr_alpha)) * (x - ((1 - curr_alpha) / (torch.sqrt(1 - curr_alpha_bar))) * predicted_noise) + torch.sqrt(curr_beta) * noise

    x = x.clamp(-1, 1) * 0.5 + 0.5
    x = x.cpu()
    
    import os
    from torchvision.utils import save_image
    
    os.makedirs(r"fake_mnist/test/fake", exist_ok=True)
    
    last = 0
    for i in range(x.size(0)):
        save_image(x[i], f"fake_mnist/test/fake/{i+last:05d}.png")
    print(i)

Generating Samples...
400
300
200
100
0
999


In [14]:
x[:, 0, :, :].shape

torch.Size([1000, 28, 28])